# Experiment H21a: Direction vs Scale Crossover -- Where Does Muon's Directional Advantage Yield to Oracle Per-Layer LR?

---

## Theoretical Background

Experiment 2.2d revealed a striking paradox: at $c = 100$ rescaling, oracle per-layer
SGD (with independently tuned LR per layer) dramatically outperforms Muon. But at
$c = 1$ (no rescaling), Muon wins by approximately 30x. This means there must exist
a **crossover point** $c^*$ where the two effects exactly balance.

### Two Competing Advantages

Muon's Newton-Schulz orthogonalization provides two distinct benefits:

1. **Scale adaptation** (per-layer LR): The Frobenius normalization $X_0 = M/\|M\|_F$
   automatically adjusts the effective learning rate per layer. This is valuable when
   layers have different gradient scales, but an oracle with per-layer LR tuning can
   replicate this.

2. **Directional quality** (polar factor): The NS iteration converges to the nearest
   orthogonal matrix, extracting the pure rotation from the gradient. This equalizes
   singular values and removes gauge degrees of freedom. An oracle per-layer LR
   *cannot* replicate this.

### The Crossover Phenomenon

At small $c$, all layers have similar gradient scales, so per-layer LR provides little
benefit. Muon's directional advantage dominates.

At large $c$, the gradient scale imbalance is extreme. The oracle's ability to
independently tune each layer's LR over a huge grid compensates for its lack of
directional quality. But Muon with a single LR cannot find the right scale for all
layers simultaneously (even though NS normalization helps, the single global LR
multiplied by the NS output may not be optimal for all layers).

## Experimental Design

Sweep $c \in \{1, 2, 5, 10, 20, 50, 100, 200, 500\}$ and for each:
- **Muon**: Sweep 10 single-LR candidates, select the best
- **Oracle per-layer SGD**: Exhaustive grid search over $5^4 = 625$ per-layer LR combinations

Compute the ratio $\text{Muon loss} / \text{Oracle loss}$. The crossover $c^*$ is
where this ratio crosses 1.0.

## Key Question

**At what rescaling factor $c^*$ does oracle per-layer LR start to beat Muon?**
This determines the boundary between the "direction-dominated" and "scale-dominated" regimes.

## Environment Setup

Import NumPy for linear algebra and itertools for the exhaustive per-layer LR grid search.

In [ ]:
import numpy as np
import itertools
import os

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))


## Experimental Configuration

4-layer deep linear network with 32x32 matrices, 300 training steps with momentum.
We use 3 seeds (fewer than usual) because the oracle grid search requires
$625 \times 3 = 1875$ training runs per $c$ value, and we sweep 9 values of $c$,
giving approximately 17,000 total training runs -- a significant computational load.

In [ ]:
DIM = 32
N_LAYERS = 4
NUM_STEPS = 300
MOMENTUM = 0.9
NS_ITERS = 5
NUM_SEEDS = 3
BATCH_SIZE = 64

print("\n--- Experimental Configuration ---")
print(f"  DIM = {DIM}")
print(f"  NUM_STEPS = {NUM_STEPS}")
print(f"  BATCH_SIZE = {BATCH_SIZE}")
print(f"  MOMENTUM = {MOMENTUM}")
print(f"  NUM_SEEDS = {NUM_SEEDS}")
print(f"  NS_ITERS = {NS_ITERS}")
print(f"  N_LAYERS = {N_LAYERS}")


In [ ]:
C_VALUES = [1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0, 200.0, 500.0]

### Rescaling Sweep and LR Grids

The $c$ values are chosen to provide fine resolution around the expected crossover
region. At $c = 1$, there is no rescaling; at $c = 500$, the norm ratio between
the first and last layer is $500^2 = 250,000$.

Muon gets 10 LR candidates to give it a fair chance at each $c$ value.
The per-layer oracle gets 5 candidates per layer spanning $10^{-1}$ to $10^{-5}$,
covering the full range of scales induced by the rescaling.

In [ ]:
MUON_LR_GRID = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.002, 0.001]

In [ ]:
# For oracle per-layer: 5 LRs per layer, 5^4 = 625 combos
PER_LAYER_LR_GRID = [0.1, 0.01, 0.001, 0.0001, 0.00001]

## Core Algorithms

### Newton-Schulz Orthogonalization
Standard NS iteration with Frobenius normalization. At each $c$ value, the implicit
per-layer LR effect of this normalization is the same, but its *sufficiency* changes:
at small $c$, all layers are similar so normalization barely matters; at large $c$,
normalization is critical but may not be enough.

### Weight Initialization
Near-identity initialization with $c$-rescaling on the first and last layers.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, 'fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

In [ ]:
def init_weights(seed, c):
    rng = np.random.RandomState(seed)
    weights = [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(N_LAYERS)]
    # Apply c-rescaling: layer 0 *= c, layer -1 *= 1/c
    weights[0] = weights[0] * c
    weights[-1] = weights[-1] / c
    return weights

## Network Architecture

Forward pass, MSE loss, and backpropagation gradient computation for the 4-layer
deep linear network. Data is generated from a random target matrix $W^*$ to provide
a well-defined learning task.

In [ ]:
def forward(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y):
    pred = forward(weights, X)
    return 0.5 * np.mean(np.sum((pred - Y) ** 2, axis=0))

In [ ]:
def compute_gradients(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X.copy()]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

In [ ]:
def make_data(seed):
    rng = np.random.RandomState(seed)
    W_target = rng.randn(DIM, DIM) * 0.5
    X = rng.randn(DIM, BATCH_SIZE) * 0.3
    Y = W_target @ X
    return X, Y

## Training Methods

### Muon Training
Muon with a single global LR. The NS step provides implicit per-layer scale adaptation
but uses the same LR multiplier across all layers.

### Oracle Per-Layer SGD
SGD with independently tuned LR per layer. This has $N_\text{layers}$ free parameters
where Muon has only 1, giving it a massive hyperparameter advantage.

In [ ]:
def train_muon(w0, X, Y, lr):
    weights = [W.copy() for W in w0]
    mom = [np.zeros_like(W) for W in weights]
    for step in range(NUM_STEPS):
        if compute_loss(weights, X, Y) > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y)
        for i in range(N_LAYERS):
            mom[i] = MOMENTUM * mom[i] + newton_schulz(grads[i])
            weights[i] -= lr * mom[i]
    fl = compute_loss(weights, X, Y)
    return fl if np.isfinite(fl) else float('inf')

In [ ]:
def train_sgd_per_layer(w0, X, Y, per_layer_lrs):
    weights = [W.copy() for W in w0]
    mom = [np.zeros_like(W) for W in weights]
    for step in range(NUM_STEPS):
        if compute_loss(weights, X, Y) > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y)
        for i in range(N_LAYERS):
            mom[i] = MOMENTUM * mom[i] + grads[i]
            weights[i] -= per_layer_lrs[i] * mom[i]
    fl = compute_loss(weights, X, Y)
    return fl if np.isfinite(fl) else float('inf')

## LR Sweep Functions

These functions perform the hyperparameter optimization for each method.
The Muon sweep tests 10 single-LR values; the oracle sweep exhaustively tests
all $5^4 = 625$ per-layer LR combinations.

In [ ]:
def sweep_muon(seeds, c):
    """Sweep LR for Muon, return best LR."""
    best_lr, best_loss = MUON_LR_GRID[-1], float('inf')
    for lr in MUON_LR_GRID:
        losses = []
        for s in seeds:
            X, Y = make_data(s)
            w = init_weights(s + 5000, c)
            fl = train_muon(w, X, Y, lr)
            losses.append(fl)
        finite = [l for l in losses if np.isfinite(l)]
        ml = np.mean(finite) if finite else float('inf')
        if ml < best_loss:
            best_loss = ml
            best_lr = lr
    return best_lr, best_loss

In [ ]:
def sweep_oracle_per_layer(seeds, c):
    """
    Full grid search: 5^4 = 625 combos. Return best per-layer LR tuple.
    """
    combos = list(itertools.product(PER_LAYER_LR_GRID, repeat=N_LAYERS))

    best_combo = None
    best_loss = float('inf')

    for combo in combos:
        lrs = list(combo)
        losses = []
        for s in seeds:
            X, Y = make_data(s)
            w = init_weights(s + 5000, c)
            fl = train_sgd_per_layer(w, X, Y, lrs)
            losses.append(fl)
        finite = [l for l in losses if np.isfinite(l)]
        ml = np.mean(finite) if finite else float('inf')
        if ml < best_loss:
            best_loss = ml
            best_combo = lrs

    return best_combo, best_loss

## Main Experiment: Crossover Sweep

The main function sweeps over all $c$ values, running both the Muon LR sweep
and the oracle grid search at each point. It then computes the Muon/Oracle loss
ratio and identifies the crossover point $c^*$.

The output includes:
- Per-$c$ loss comparison and regime classification (direction vs scale)
- Crossover point detection by sign change of $\log(\text{ratio})$
- Power-law fit: $\text{ratio} \sim c^\alpha$ to extrapolate the crossover
- Detailed analysis at $c = 1$ to quantify Muon's pure directional advantage

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H21a: DIRECTION vs SCALE CROSSOVER")
    print("=" * 100)
    print(f"Network: {N_LAYERS}-layer deep linear {DIM}x{DIM}")
    print(f"c values: {C_VALUES}")
    print(f"Steps: {NUM_STEPS}, Seeds: {NUM_SEEDS}")
    print(f"Oracle grid: {len(PER_LAYER_LR_GRID)}^{N_LAYERS} = "
          f"{len(PER_LAYER_LR_GRID) ** N_LAYERS} combos")
    print()

    results = {}

    for c in C_VALUES:
        print(f"\n  c={c:.0f}")

        # Muon LR sweep
        best_muon_lr, muon_sweep_loss = sweep_muon(seeds, c)
        print(f"    Muon best LR: {best_muon_lr}, sweep loss: {muon_sweep_loss:.4e}")

        # Oracle per-layer LR
        print(f"    Oracle grid search (625 combos)...", end=" ", flush=True)
        best_oracle_lrs, oracle_sweep_loss = sweep_oracle_per_layer(seeds, c)
        print(f"done. Best LRs: {[f'{lr:.0e}' for lr in best_oracle_lrs]}")

        # Full evaluation
        muon_losses = []
        oracle_losses = []
        for s in seeds:
            X, Y = make_data(s)
            w = init_weights(s + 5000, c)
            muon_losses.append(train_muon(w, X, Y, best_muon_lr))
            oracle_losses.append(train_sgd_per_layer(w, X, Y, best_oracle_lrs))

        muon_mean = np.mean([l for l in muon_losses if np.isfinite(l)] or [float('inf')])
        oracle_mean = np.mean([l for l in oracle_losses if np.isfinite(l)] or [float('inf')])

        ratio = muon_mean / max(oracle_mean, 1e-30)
        regime = "DIRECTION (Muon wins)" if ratio < 1.0 else "SCALE (Oracle wins)"

        results[c] = {
            'muon': muon_mean, 'oracle': oracle_mean,
            'ratio': ratio, 'regime': regime,
            'muon_lr': best_muon_lr, 'oracle_lrs': best_oracle_lrs,
        }
        print(f"    Muon={muon_mean:.4e}  Oracle={oracle_mean:.4e}  "
              f"Muon/Oracle={ratio:.4f}  [{regime}]")

    # =========================================================================
    # RESULTS
    # =========================================================================
    print(f"\n\n{'=' * 100}")
    print("RESULTS: DIRECTION vs SCALE CROSSOVER")
    print(f"{'=' * 100}")

    print(f"\n  {'c':>6}  {'Muon loss':>12}  {'Oracle loss':>12}  {'Muon/Oracle':>12}  {'Regime':>25}")
    print("  " + "-" * 75)

    c_star = None
    prev_ratio = None
    for c in C_VALUES:
        r = results[c]
        marker = ""
        if prev_ratio is not None and prev_ratio < 1.0 and r['ratio'] >= 1.0 and c_star is None:
            c_star = c
            marker = "  <-- crossover"
        elif prev_ratio is not None and prev_ratio >= 1.0 and r['ratio'] < 1.0 and c_star is None:
            c_star = c
            marker = "  <-- crossover"
        prev_ratio = r['ratio']

        print(f"  {c:>6.0f}  {r['muon']:>12.4e}  {r['oracle']:>12.4e}  "
              f"{r['ratio']:>12.4f}  {r['regime']:>25}{marker}")

    # If no crossover found, check if it's all one regime
    if c_star is None:
        ratios = [results[c]['ratio'] for c in C_VALUES]
        if all(r < 1.0 for r in ratios):
            print(f"\n  Muon wins at ALL c values tested. c* > {C_VALUES[-1]}")
        elif all(r >= 1.0 for r in ratios):
            print(f"\n  Oracle wins at ALL c values tested. c* < {C_VALUES[0]}")
        else:
            # Find the transition
            for i in range(1, len(C_VALUES)):
                r_prev = results[C_VALUES[i - 1]]['ratio']
                r_curr = results[C_VALUES[i]]['ratio']
                if (r_prev < 1.0) != (r_curr < 1.0):
                    c_star = (C_VALUES[i - 1] + C_VALUES[i]) / 2
                    break
            print(f"\n  Estimated c* (crossover): ~{c_star:.0f}")
    else:
        print(f"\n  Crossover c* = {c_star:.0f}")

    # Detailed analysis at c=1 (direction regime)
    print(f"\n  === Analysis at c=1 (direction regime) ===")
    r1 = results[1.0]
    print(f"    Muon loss:   {r1['muon']:.4e}")
    print(f"    Oracle loss: {r1['oracle']:.4e}")
    print(f"    Muon/Oracle: {r1['ratio']:.4f}")
    if r1['ratio'] < 1.0:
        print(f"    Muon's DIRECTIONAL advantage: {1.0 / r1['ratio']:.1f}x better than oracle per-layer LR")
    else:
        print(f"    Oracle SGD is already better even at c=1")

    # Trend analysis
    print(f"\n  === Trend: Muon/Oracle ratio vs c ===")
    cs = np.array(C_VALUES)
    ratios = np.array([results[c]['ratio'] for c in C_VALUES])
    log_cs = np.log10(cs)
    log_ratios = np.log10(np.clip(ratios, 1e-30, None))
    if len(log_cs) >= 3:
        slope, intercept = np.polyfit(log_cs, log_ratios, 1)
        print(f"    log10(ratio) = {slope:.3f} * log10(c) + {intercept:.3f}")
        print(f"    Power law: ratio ~ c^{slope:.3f}")
        if slope > 0:
            # Find where ratio = 1 => log_ratio = 0
            c_star_fit = 10 ** (-intercept / slope)
            print(f"    Extrapolated c* from power law: {c_star_fit:.1f}")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

In [ ]:
if __name__ == '__main__':
    main()

## Conclusions

### What This Experiment Reveals

This experiment maps the **phase diagram** of Muon's advantage as a function of
reparametrization severity. The crossover point $c^*$ is a concrete, measurable
number that quantifies where Muon's directional benefit gives way to scale-tuning
limitations.

### Interpretation Guide

- **Low $c^*$ (e.g., $c^* < 10$)**: Muon's directional advantage is weak and easily
  overwhelmed by per-layer LR tuning. This would suggest the polar factor provides
  marginal benefit and Muon is primarily a convenience optimizer.

- **High $c^*$ (e.g., $c^* > 100$)**: Muon's directional advantage is robust and
  dominates across the practical operating regime. Only extreme rescalings, unlikely
  in real networks, can make oracle per-layer SGD competitive.

- **No crossover found (Muon wins everywhere)**: The directional quality of the polar
  factor is so beneficial that even a 625-combination oracle grid search cannot compensate.
  This would be the strongest evidence for Muon's gauge-fixing mechanism.

### Power-Law Scaling

If the ratio follows $\text{Muon/Oracle} \sim c^\alpha$:
- $\alpha > 0$: Muon's relative performance degrades with increasing $c$
  (expected, since single-LR Muon cannot perfectly adapt to extreme scale imbalances)
- $\alpha \approx 0$: Ratio is scale-invariant (surprising, would suggest Muon's
  NS normalization perfectly compensates)
- Large $\alpha$ (e.g., > 1): Rapid degradation, suggesting Muon's single-LR is
  a significant limitation under rescaling

### Connection to the Framework

- **Exp 2.2d**: Established the paradox this experiment resolves
- **H3/H3a**: Direction quality measurements at $c = 1$
- **H6a**: LR confound audit -- related concern about LR artifacts
- **H21b**: Next experiment, testing whether the balance between direction and scale
  can be predicted from spectral properties